In [5]:
# Load and prepare all Dataset A raw events
from pathlib import Path
import json
import pandas as pd
PROJECT_ROOT = Path("..")
DATASET_A_DIR = PROJECT_ROOT / "notebooks" / "dataset_A" / "dataset_a_combined"
DATASET_B_DIR = PROJECT_ROOT / "notebooks" / "dataset_B"

def load_jsonl_events(events_file):
    """
    Load one events.jsonl file into a list of dictionaries.
    Invalid/empty lines are skipped safely.
    """
    records = []

    with open(events_file, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                print(
                    f"Warning: could not parse line {line_number} "
                    f"in {events_file}"
                )

    return records


# Find every events.jsonl file in Dataset A


event_files_a = sorted(DATASET_A_DIR.rglob("events.jsonl"))

print("=" * 70)
print("DATASET A — RAW EVENT LOADING")
print("=" * 70)

print(f"Event files found: {len(event_files_a)}")

#Load all events

all_event_records = []

for event_file in event_files_a:
    records = load_jsonl_events(event_file)

    all_event_records.extend(records)


print(f"Raw events loaded: {len(all_event_records):,}")



# Convert to DataFrame


ALL_EVENTS_A = pd.json_normalize(all_event_records)


# Sort chronologically


if "timestamp_ms" not in ALL_EVENTS_A.columns:
    raise ValueError(
        "timestamp_ms column was not found in the raw event data."
    )

ALL_EVENTS_A = (
    ALL_EVENTS_A
    .sort_values(
        by=["session_id", "timestamp_ms"],
        kind="stable"
    )
    .reset_index(drop=True)
)



# Basic information


print("\nDataFrame shape:")
print(ALL_EVENTS_A.shape)

print("\nSessions:")
print(ALL_EVENTS_A["session_id"].nunique())

print("\nEvent types:")
print(ALL_EVENTS_A["event_type"].value_counts().head(15))

print("\nFirst 5 chronological events:")
print(
    ALL_EVENTS_A[
        [
            "session_id",
            "timestamp_ms",
            "timestamp_iso",
            "event_type"
        ]
    ].head().to_string(index=False)
)

DATASET A — RAW EVENT LOADING
Event files found: 117
Raw events loaded: 162,768

DataFrame shape:
(162768, 410)

Sessions:
63

Event types:
event_type
app_switch                50588
keystroke                 38717
screenshot_smart          34580
shortcut                  13668
mouse_click                6177
browser_click              5365
clipboard_change           5198
mouse_scroll               3180
browser_form_input         1805
browser_navigation         1725
window_title_change         451
window_state_change         232
browser_error               200
extension_disconnected      172
browser_alert               137
Name: count, dtype: int64

First 5 chronological events:
                         session_id  timestamp_ms            timestamp_iso       event_type
ses_20260630-121953-LAPTOP-R36BQBTE 1782821993821 2026-06-30T12:19:53.821Z    session_start
ses_20260630-121953-LAPTOP-R36BQBTE 1782821994435 2026-06-30T12:19:54.435Z      mouse_click
ses_20260630-121953-LAPTOP-R36BQBTE 

In [6]:
#Inspect the actual Dataset A ground-truth schema

import json
from collections import Counter

print("=" * 70)
print("DATASET A — GROUND TRUTH SCHEMA INSPECTION")
print("=" * 70)


# 1. Find GT files


gt_files = sorted(DATASET_A_DIR.rglob("gt.jsonl"))

print(f"\nGT files found: {len(gt_files)}")



# 2. Load all GT records

all_gt_records = []

for gt_file in gt_files:

    with open(gt_file, "r", encoding="utf-8") as f:

        for line_number, line in enumerate(f, start=1):

            line = line.strip()

            if not line:
                continue

            try:
                all_gt_records.append(json.loads(line))

            except json.JSONDecodeError:
                print(
                    f"Warning: invalid JSON in {gt_file}, "
                    f"line {line_number}"
                )

print(f"Total GT records loaded: {len(all_gt_records):,}")



# 3. Inspect the actual top-level keys


print("\nTOP-LEVEL KEYS")
print("--------------")

key_counts = Counter()

for record in all_gt_records:
    key_counts.update(record.keys())

for key, count in key_counts.most_common():
    print(str(key), "->", count)



# 4. Show first 5 records


print("\nFIRST 5 GT RECORDS")
print("==================")

for i, record in enumerate(all_gt_records[:5], start=1):

    print(f"\n--- Record {i} ---")

    print(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        )[:3000]
    )

DATASET A — GROUND TRUTH SCHEMA INSPECTION

GT files found: 63
Total GT records loaded: 10,609

TOP-LEVEL KEYS
--------------
ts_utc -> 10609
run_id -> 10609
event -> 10609
current_process -> 10546
note_id -> 4776
process_variant -> 4744
target_app -> 3010
target_field -> 3010
process_code -> 2009
process_name -> 2009
case_id -> 2009
task_id -> 2009
action -> 2009
entity -> 2009
source_app -> 1766
content_preview -> 1766
from -> 1689
to -> 1689
split_id -> 289
phase -> 190
seed -> 63
dwell_scale -> 63
n_procs -> 63
run_date -> 63
operator -> 63
operator_dept -> 63
machine_id -> 63
noise_rate -> 63
noise_blocks -> 63
chunk_split -> 63
continuation_family -> 63
continuation_dwell -> 63
split_min_gap -> 63
duration -> 63
domain_mix -> 63
transition -> 63
operator_type -> 63
tasks_per_proc -> 63
selected_procs -> 63
split_procs -> 63
session_notes -> 63
total_tasks -> 63
duration_seconds -> 63

FIRST 5 GT RECORDS

--- Record 1 ---
{
  "ts_utc": "2026-06-30T12:20:25.140811+00:00",
  "run_id

In [8]:
#Analyze GT event types and process-related records

from collections import Counter
import pandas as pd

print("=" * 70)
print("DATASET A — GT EVENT SEMANTICS")
print("=" * 70)


# 1. Count all GT event types


event_counts = Counter(
    record.get("event")
    for record in all_gt_records
)

print("\nGT EVENT COUNTS")
print("----------------")

for event_name, count in event_counts.most_common():
    print(f"{str(event_name):<30} {count:>7,}")



# 2. Inspect process-related event types


process_events = {
    "process_started",
    "process_switched_out",
    "process_suspended",
    "process_resumed",
    "task_started",
}

print("\nPROCESS-RELATED EVENT COUNTS")
print("----------------------------")

for event_name in process_events:
    print(
        f"{event_name:<30} "
        f"{event_counts.get(event_name, 0):>7,}"
    )



# 3. Inspect the structure of each process-related event


print("\nPROCESS-RELATED RECORD EXAMPLES")
print("================================")

for event_name in process_events:

    matching_records = [
        record
        for record in all_gt_records
        if record.get("event") == event_name
    ]

    print(f"\n--- {event_name} ---")
    print(f"Number of records: {len(matching_records)}")

    if matching_records:

        # Show first record
        record = matching_records[0]

        print("\nExample:")
        for key, value in record.items():
            print(f"  {key}: {value}")



# 4. Compare fields populated by each process event type


print("\nFIELD AVAILABILITY BY PROCESS EVENT")
print("===================================")

fields_to_check = [
    "ts_utc",
    "current_process",
    "process_variant",
    "process_code",
    "process_name",
    "case_id",
    "task_id",
    "action",
    "entity",
    "source_app",
    "target_app",
    "target_field",
    "note_id",
    "content_preview",
]

for event_name in process_events:

    records = [
        record
        for record in all_gt_records
        if record.get("event") == event_name
    ]

    if not records:
        continue

    print(f"\n--- {event_name} ---")

    for field in fields_to_check:

        populated = sum(
            record.get(field) is not None
            for record in records
        )

        percentage = 100 * populated / len(records)

        print(
            f"{field:<20} "
            f"{populated:>5}/{len(records):<5} "
            f"({percentage:>6.2f}%)"
        )


# 5. Create a compact GT dataframe


GT_RAW = pd.DataFrame(all_gt_records)

print("\n\nGT DATAFRAME")
print("============")

print("Shape:", GT_RAW.shape)

print("\nColumns:")
print(list(GT_RAW.columns))

print("\nFirst 5 process_started records:")

print(
    GT_RAW[
        GT_RAW["event"] == "process_started"
    ][
        [
            "ts_utc",
            "event",
            "current_process",
            "process_code",
            "process_name",
            "case_id",
            "task_id"
        ]
    ].head().to_string(index=False)
)

DATASET A — GT EVENT SEMANTICS

GT EVENT COUNTS
----------------
clipboard_paste                  3,010
task_started                     2,009
process_started                  1,819
clipboard_copy                   1,766
process_switched_out             1,590
process_resumed                    190
process_suspended                   99
run_config                          63
session_ended                       63

PROCESS-RELATED EVENT COUNTS
----------------------------
process_suspended                   99
process_resumed                    190
task_started                     2,009
process_switched_out             1,590
process_started                  1,819

PROCESS-RELATED RECORD EXAMPLES

--- process_suspended ---
Number of records: 99

Example:
  ts_utc: 2026-06-30T12:25:11.308831+00:00
  run_id: theme_m1_20260630_175009
  event: process_suspended
  current_process: B
  process_variant: reg
  from: B
  to: H
  split_id: M1-B-split-1

--- process_resumed ---
Number of records: 19

In [9]:
#Inspect process lifecycle events in chronological order

print("=" * 70)
print("DATASET A — PROCESS LIFECYCLE INSPECTION")
print("=" * 70)


# Select one representative session


inspection_session = sorted(
    ALL_EVENTS_A["session_id"].unique()
)[0]

print(f"\nInspection session:")
print(inspection_session)



# Get GT records for this session


session_gt = GT_RAW[
    GT_RAW["run_id"].notna()
].copy()

# The GT records do not necessarily use session_id directly.
# Use the run_id/session mapping available in the GT and inspect
# the first session's records through timestamp/process context.

# Instead, identify the first GT file corresponding to the
# first Dataset A session.

first_session_dir = sorted(
    [
        p for p in DATASET_A_DIR.iterdir()
        if p.is_dir() and p.name.startswith("ses_")
    ]
)[0]

first_gt_file = first_session_dir / "gt.jsonl"

session_records = []

with open(first_gt_file, "r", encoding="utf-8") as f:

    for line in f:

        line = line.strip()

        if line:
            session_records.append(json.loads(line))


session_gt = pd.DataFrame(session_records)



# Keep only process lifecycle events


lifecycle_events = [
    "process_started",
    "process_switched_out",
    "process_suspended",
    "process_resumed"
]

lifecycle_gt = session_gt[
    session_gt["event"].isin(lifecycle_events)
].copy()


# Sort chronologically


lifecycle_gt["ts_utc_parsed"] = pd.to_datetime(
    lifecycle_gt["ts_utc"],
    utc=True,
    errors="coerce"
)

lifecycle_gt = lifecycle_gt.sort_values(
    "ts_utc_parsed"
).reset_index(drop=True)


# Display important lifecycle information


display_columns = [
    "ts_utc",
    "event",
    "current_process",
    "process_code",
    "process_name",
    "case_id",
    "process_variant"
]

available_columns = [
    col for col in display_columns
    if col in lifecycle_gt.columns
]

print("\nPROCESS LIFECYCLE TIMELINE")
print("=========================")

print(
    lifecycle_gt[available_columns]
    .to_string(index=False)
)


# Show unique values of current_process by event type


print("\n\nCURRENT PROCESS VALUES BY EVENT TYPE")
print("====================================")

for event_name in lifecycle_events:

    subset = lifecycle_gt[
        lifecycle_gt["event"] == event_name
    ]

    if len(subset) == 0:
        continue

    values = (
        subset["current_process"]
        .dropna()
        .astype(str)
        .value_counts()
    )

    print(f"\n--- {event_name} ---")
    print(values.to_string())



# Show records around the first few process switches


print("\n\nFIRST PROCESS SWITCHES WITH CONTEXT")
print("===================================")

switch_indices = lifecycle_gt.index[
    lifecycle_gt["event"] == "process_switched_out"
].tolist()

for idx in switch_indices[:5]:

    start_idx = max(0, idx - 2)
    end_idx = min(len(lifecycle_gt), idx + 3)

    print(f"\n--- Context around lifecycle row {idx} ---")

    print(
        lifecycle_gt
        .iloc[start_idx:end_idx][available_columns]
        .to_string(index=False)
    )

DATASET A — PROCESS LIFECYCLE INSPECTION

Inspection session:
ses_20260630-121953-LAPTOP-R36BQBTE

PROCESS LIFECYCLE TIMELINE
                          ts_utc                event current_process process_code process_name        case_id process_variant
2026-06-30T12:21:12.794418+00:00      process_started               H            H       銀行勘定照合  BR-175009-001             NaN
2026-06-30T12:21:41.332221+00:00 process_switched_out               H          NaN          NaN            NaN             NaN
2026-06-30T12:21:41.335282+00:00      process_started               C            C    育児・産休申請確認  LA-175009-001             NaN
2026-06-30T12:22:25.680580+00:00 process_switched_out               C          NaN          NaN            NaN             NaN
2026-06-30T12:22:25.682643+00:00      process_started               B            B    給与備考・控除整備  PI-175009-001             reg
2026-06-30T12:23:22.089303+00:00 process_switched_out               B          NaN          NaN            NaN  

In [10]:
#Inspect suspension and resumption relationships

print("=" * 70)
print("DATASET A — SUSPENSION / RESUMPTION ANALYSIS")
print("=" * 70)


# 1. Extract suspension and resumption records


suspensions = lifecycle_gt[
    lifecycle_gt["event"] == "process_suspended"
].copy()

resumptions = lifecycle_gt[
    lifecycle_gt["event"] == "process_resumed"
].copy()


print("\nSuspensions in inspected session:", len(suspensions))
print("Resumptions in inspected session:", len(resumptions))



# 2. Display all suspension records


print("\nSUSPENSION RECORDS")
print("==================")

suspension_columns = [
    "ts_utc",
    "event",
    "current_process",
    "process_code",
    "process_name",
    "case_id",
    "process_variant"
]

available_suspension_columns = [
    col for col in suspension_columns
    if col in suspensions.columns
]

if len(suspensions) > 0:
    print(
        suspensions[
            available_suspension_columns
        ].to_string(index=False)
    )
else:
    print("No suspension records in this session.")



# 3. Display all resumption records


print("\n\nRESUMPTION RECORDS")
print("==================")

if len(resumptions) > 0:
    print(
        resumptions[
            available_suspension_columns
        ].to_string(index=False)
    )
else:
    print("No resumption records in this session.")


# 4. Compare process identities


print("\n\nPROCESS VALUES")
print("==============")

print("\nSuspended process values:")

if len(suspensions) > 0:
    print(
        suspensions[
            ["current_process", "process_code", "case_id"]
        ].to_string(index=False)
    )

print("\nResumed process values:")

if len(resumptions) > 0:
    print(
        resumptions[
            ["current_process", "process_code", "case_id"]
        ].to_string(index=False)
    )



# 5. Inspect temporal ordering around each suspension


print("\n\nSUSPENSION → FOLLOWING GT EVENTS")
print("================================")

for idx in suspensions.index:

    position = lifecycle_gt.index.get_loc(idx)

    start_idx = max(0, position - 1)
    end_idx = min(
        len(lifecycle_gt),
        position + 4
    )

    print(f"\n--- Around suspension at row {idx} ---")

    print(
        lifecycle_gt
        .iloc[start_idx:end_idx][available_suspension_columns]
        .to_string(index=False)
    )



# 6. Inspect temporal ordering around each resumption


print("\n\nRESUMPTION → FOLLOWING GT EVENTS")
print("================================")

for idx in resumptions.index:

    position = lifecycle_gt.index.get_loc(idx)

    start_idx = max(0, position - 2)
    end_idx = min(
        len(lifecycle_gt),
        position + 3
    )

    print(f"\n--- Around resumption at row {idx} ---")

    print(
        lifecycle_gt
        .iloc[start_idx:end_idx][available_suspension_columns]
        .to_string(index=False)
    )

DATASET A — SUSPENSION / RESUMPTION ANALYSIS

Suspensions in inspected session: 1
Resumptions in inspected session: 1

SUSPENSION RECORDS
                          ts_utc             event current_process process_code process_name case_id process_variant
2026-06-30T12:25:11.308831+00:00 process_suspended               B          NaN          NaN     NaN             reg


RESUMPTION RECORDS
                          ts_utc           event current_process process_code process_name       case_id process_variant
2026-06-30T12:30:59.741103+00:00 process_resumed               B            B    給与備考・控除整備 PI-175009-003             reg


PROCESS VALUES

Suspended process values:
current_process process_code case_id
              B          NaN     NaN

Resumed process values:
current_process process_code       case_id
              B            B PI-175009-003


SUSPENSION → FOLLOWING GT EVENTS

--- Around suspension at row 13 ---
                          ts_utc                event current_pr

In [11]:
#Global analysis of process cases and lifecycle events

print("=" * 70)
print("DATASET A — PROCESS CASE / LIFECYCLE ANALYSIS")
print("=" * 70)



# 1. Basic case statistics


process_records = GT_RAW[
    GT_RAW["event"].isin([
        "process_started",
        "process_switched_out",
        "process_suspended",
        "process_resumed"
    ])
].copy()

print("\nPROCESS LIFECYCLE RECORDS")
print("------------------------")
print(f"Total lifecycle records: {len(process_records):,}")

print(
    f"Records with case_id: "
    f"{process_records['case_id'].notna().sum():,}"
)

print(
    f"Records without case_id: "
    f"{process_records['case_id'].isna().sum():,}"
)


# 2. Case IDs from process_started


started = GT_RAW[
    GT_RAW["event"] == "process_started"
].copy()

print("\nPROCESS START CASE IDs")
print("----------------------")

print(
    f"Process starts: {len(started):,}"
)

print(
    f"Starts with case_id: "
    f"{started['case_id'].notna().sum():,}"
)

print(
    f"Unique case IDs: "
    f"{started['case_id'].nunique(dropna=True):,}"
)



# 3. Number of starts per case


starts_per_case = (
    started
    .dropna(subset=["case_id"])
    .groupby("case_id")
    .size()
    .sort_values(ascending=False)
)

print("\nSTARTS PER CASE")
print("---------------")

print(
    f"Cases appearing more than once in process_started: "
    f"{(starts_per_case > 1).sum()}"
)

print("\nTop repeated case IDs:")

if len(starts_per_case) > 0:
    print(
        starts_per_case[
            starts_per_case > 1
        ].head(20).to_string()
    )
else:
    print("None")



# 4. Case availability by lifecycle event


print("\nCASE-ID AVAILABILITY BY EVENT")
print("-----------------------------")

for event_name in [
    "process_started",
    "process_switched_out",
    "process_suspended",
    "process_resumed"
]:

    subset = GT_RAW[
        GT_RAW["event"] == event_name
    ]

    total = len(subset)

    with_case = subset["case_id"].notna().sum()

    percentage = (
        100 * with_case / total
        if total > 0
        else 0
    )

    print(
        f"{event_name:<25} "
        f"{with_case:>5}/{total:<5} "
        f"({percentage:>6.2f}%)"
    )


# 5. Process identity by case


print("\nPROCESS CODE CONSISTENCY WITHIN CASE")
print("------------------------------------")

case_process_counts = (
    started
    .dropna(subset=["case_id"])
    .groupby("case_id")["process_code"]
    .nunique()
)

print(
    f"Cases with exactly one process code: "
    f"{(case_process_counts == 1).sum():,}"
)

print(
    f"Cases with multiple process codes: "
    f"{(case_process_counts > 1).sum():,}"
)


# 6. Inspect suspension/resumption case relationships


suspended = GT_RAW[
    GT_RAW["event"] == "process_suspended"
].copy()

resumed = GT_RAW[
    GT_RAW["event"] == "process_resumed"
].copy()

print("\nSUSPENSION / RESUMPTION CASE INFORMATION")
print("-----------------------------------------")

print(
    f"Suspensions with case_id: "
    f"{suspended['case_id'].notna().sum()}/{len(suspended)}"
)

print(
    f"Resumptions with case_id: "
    f"{resumed['case_id'].notna().sum()}/{len(resumed)}"
)



# 7. Show all resumption records


print("\nRESUMPTION RECORDS — PROCESS / CASE")
print("-----------------------------------")

if len(resumed) > 0:

    columns = [
        "ts_utc",
        "current_process",
        "process_code",
        "case_id",
        "process_variant"
    ]

    print(
        resumed[columns]
        .sort_values("ts_utc")
        .head(30)
        .to_string(index=False)
    )

else:

    print("No resumption records found.")



# 8. Check whether process_started cases are unique


print("\nCASE-ID UNIQUENESS CHECK")
print("------------------------")

duplicate_case_ids = (
    starts[
        starts["case_id"].duplicated(keep=False)
    ]
    .sort_values("case_id")
)

if len(duplicate_case_ids) == 0:

    print(
        "Every process_started record has a unique case_id."
    )

else:

    print(
        f"Found {len(duplicate_case_ids)} "
        f"process_started records sharing case IDs."
    )

    print(
        duplicate_case_ids[
            [
                "ts_utc",
                "process_code",
                "process_name",
                "case_id"
            ]
        ]
        .head(20)
        .to_string(index=False)
    )

DATASET A — PROCESS CASE / LIFECYCLE ANALYSIS

PROCESS LIFECYCLE RECORDS
------------------------
Total lifecycle records: 3,698
Records with case_id: 2,009
Records without case_id: 1,689

PROCESS START CASE IDs
----------------------
Process starts: 1,819
Starts with case_id: 1,819
Unique case IDs: 1,793

STARTS PER CASE
---------------
Cases appearing more than once in process_started: 26

Top repeated case IDs:
case_id
BV-111912-003     2
EXP-111912-003    2
EXP-111912-004    2
INV-111912-002    2
BR-111912-001     2
BR-111912-002     2
BR-111912-003     2
BR-111912-004     2
INV-111912-003    2
INV-111912-001    2
EXP-111912-001    2
EXP-111912-002    2
BV-111912-001     2
BV-111912-002     2
PM-111912-001     2
PM-111912-002     2
PM-111912-003     2
RT-111912-001     2
RT-111912-002     2
RT-111912-003     2

CASE-ID AVAILABILITY BY EVENT
-----------------------------
process_started            1819/1819  (100.00%)
process_switched_out          0/1590  (  0.00%)
process_suspended

NameError: name 'starts' is not defined

In [12]:
#  Inspect repeated case IDs in process_started

print("=" * 70)
print("DATASET A — REPEATED PROCESS CASE IDs")
print("=" * 70)


# Get process_started records


started = GT_RAW[
    GT_RAW["event"] == "process_started"
].copy()



# Count process starts per case


case_counts = (
    started["case_id"]
    .dropna()
    .value_counts()
)

duplicate_cases = case_counts[
    case_counts > 1
]


print(
    f"\nUnique duplicated case IDs: "
    f"{len(duplicate_cases)}"
)

print(
    f"Total process_started records "
    f"belonging to duplicated cases: "
    f"{duplicate_cases.sum()}"
)



# Display duplicated case IDs


print("\nDUPLICATED CASE IDs")
print("-------------------")

print(
    duplicate_cases.to_string()
)


# Get actual records for duplicated cases


duplicate_case_list = duplicate_cases.index.tolist()

duplicate_records = (
    started[
        started["case_id"].isin(duplicate_case_list)
    ]
    .sort_values(
        ["case_id", "ts_utc"]
    )
)



# Display details


print("\n\nDETAILS OF DUPLICATED CASES")
print("===========================")

columns = [
    "ts_utc",
    "current_process",
    "process_code",
    "process_name",
    "case_id",
    "process_variant"
]

print(
    duplicate_records[columns]
    .to_string(index=False)
)



# Check process consistency


print("\n\nPROCESS CONSISTENCY")
print("===================")

for case_id, group in duplicate_records.groupby("case_id"):

    process_codes = (
        group["process_code"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    process_names = (
        group["process_name"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    print(f"\nCase ID: {case_id}")
    print(f"Process codes: {process_codes}")
    print(f"Process names: {process_names}")

    print(
        group[
            [
                "ts_utc",
                "process_code",
                "process_name",
                "process_variant"
            ]
        ].to_string(index=False)
    )

DATASET A — REPEATED PROCESS CASE IDs

Unique duplicated case IDs: 26
Total process_started records belonging to duplicated cases: 52

DUPLICATED CASE IDs
-------------------
case_id
INV-111912-001    2
RT-111912-001     2
INV-111912-002    2
STK-111912-001    2
BR-111912-001     2
RT-111912-002     2
BV-111912-001     2
STK-111912-002    2
BR-111912-002     2
RT-111912-003     2
EXP-111912-001    2
EXP-111912-002    2
INV-111912-003    2
BR-111912-003     2
EXP-111912-003    2
PM-111912-001     2
RT-111912-004     2
BV-111912-002     2
PM-111912-002     2
BR-111912-004     2
EXP-111912-004    2
PM-111912-003     2
SHP-111912-001    2
SHP-111912-002    2
BV-111912-003     2
SHP-111912-003    2


DETAILS OF DUPLICATED CASES
                          ts_utc current_process process_code process_name        case_id process_variant
2026-07-01T05:52:49.106388+00:00               H            H       銀行勘定照合  BR-111912-001             NaN
2026-07-01T05:52:49.106388+00:00               H       

In [13]:
# Cell 10 — Validate whether repeated case IDs are true duplicates

print("=" * 70)
print("DATASET A — DUPLICATE CASE VALIDATION")
print("=" * 70)


# ------------------------------------------------------------
# Focus only on process_started records
# ------------------------------------------------------------

started = GT_RAW[
    GT_RAW["event"] == "process_started"
].copy()


# ------------------------------------------------------------
# Find duplicated process_started rows
# ------------------------------------------------------------

duplicate_mask = started.duplicated(
    keep=False
)

exact_duplicate_rows = started[
    duplicate_mask
].copy()


print("\nEXACT DUPLICATE ROW ANALYSIS")
print("----------------------------")

print(
    f"Exact duplicate process_started rows: "
    f"{len(exact_duplicate_rows)}"
)


# ------------------------------------------------------------
# Check duplicated rows after excluding timestamp
# ------------------------------------------------------------

identity_columns = [
    "case_id",
    "process_code",
    "process_name",
    "process_variant"
]

identity_columns = [
    col
    for col in identity_columns
    if col in started.columns
]


duplicate_identity_rows = started[
    started.duplicated(
        subset=identity_columns,
        keep=False
    )
].copy()


print(
    f"\nRows sharing the same case/process identity: "
    f"{len(duplicate_identity_rows)}"
)


# ------------------------------------------------------------
# Display all duplicated process starts
# ------------------------------------------------------------

print("\n\nDUPLICATED PROCESS START RECORDS")
print("===============================")

display_columns = [
    "ts_utc",
    "current_process",
    "process_code",
    "process_name",
    "case_id",
    "process_variant"
]

display_columns = [
    col
    for col in display_columns
    if col in started.columns
]


print(
    duplicate_identity_rows[
        display_columns
    ]
    .sort_values(
        ["case_id", "ts_utc"]
    )
    .to_string(index=False)
)


# ------------------------------------------------------------
# For each duplicated case, compare timestamps
# ------------------------------------------------------------

print("\n\nTIMESTAMP DIFFERENCES WITHIN DUPLICATED CASES")
print("=============================================")

for case_id, group in duplicate_identity_rows.groupby("case_id"):

    timestamps = pd.to_datetime(
        group["ts_utc"],
        utc=True,
        errors="coerce"
    ).sort_values()

    if len(timestamps) > 1:

        differences = (
            timestamps.diff()
            .dropna()
            .dt.total_seconds()
        )

        print(
            f"\n{case_id}: "
            f"{len(group)} starts | "
            f"timestamp gaps = "
            f"{differences.tolist()}"
        )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n\nSUMMARY")
print("=======")

print(
    f"Total process_started records: "
    f"{len(started):,}"
)

print(
    f"Unique case IDs: "
    f"{started['case_id'].nunique(dropna=True):,}"
)

print(
    f"Rows that are exact duplicates: "
    f"{len(exact_duplicate_rows):,}"
)

print(
    f"Rows involved in repeated case identities: "
    f"{len(duplicate_identity_rows):,}"
)

DATASET A — DUPLICATE CASE VALIDATION

EXACT DUPLICATE ROW ANALYSIS
----------------------------
Exact duplicate process_started rows: 52

Rows sharing the same case/process identity: 52


DUPLICATED PROCESS START RECORDS
                          ts_utc current_process process_code process_name        case_id process_variant
2026-07-01T05:52:49.106388+00:00               H            H       銀行勘定照合  BR-111912-001             NaN
2026-07-01T05:52:49.106388+00:00               H            H       銀行勘定照合  BR-111912-001             NaN
2026-07-01T05:55:45.906952+00:00               H            H       銀行勘定照合  BR-111912-002             NaN
2026-07-01T05:55:45.906952+00:00               H            H       銀行勘定照合  BR-111912-002             NaN
2026-07-01T05:57:51.777142+00:00               H            H       銀行勘定照合  BR-111912-003             NaN
2026-07-01T05:57:51.777142+00:00               H            H       銀行勘定照合  BR-111912-003             NaN
2026-07-01T06:05:09.259321+00:00    

In [14]:
# Cell 11 — Build clean process-start reference table

print("=" * 70)
print("DATASET A — CLEAN PROCESS-START REFERENCE")
print("=" * 70)


# ------------------------------------------------------------
# Select process_started records
# ------------------------------------------------------------

GT_PROCESS_STARTS = GT_RAW[
    GT_RAW["event"] == "process_started"
].copy()


# ------------------------------------------------------------
# Remove exact duplicate records
# ------------------------------------------------------------

before_dedup = len(GT_PROCESS_STARTS)

GT_PROCESS_STARTS = (
    GT_PROCESS_STARTS
    .drop_duplicates()
    .copy()
)

after_dedup = len(GT_PROCESS_STARTS)


# ------------------------------------------------------------
# Parse timestamp
# ------------------------------------------------------------

GT_PROCESS_STARTS["timestamp"] = pd.to_datetime(
    GT_PROCESS_STARTS["ts_utc"],
    utc=True,
    errors="coerce"
)


# ------------------------------------------------------------
# Keep only fields needed for segmentation reference
# ------------------------------------------------------------

reference_columns = [
    "run_id",
    "ts_utc",
    "timestamp",
    "event",
    "current_process",
    "process_code",
    "process_name",
    "case_id",
    "process_variant"
]

reference_columns = [
    column
    for column in reference_columns
    if column in GT_PROCESS_STARTS.columns
]

GT_PROCESS_STARTS = (
    GT_PROCESS_STARTS[
        reference_columns
    ]
    .sort_values(
        ["run_id", "timestamp"],
        kind="stable"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nPROCESS-START CLEANING")
print("----------------------")

print(
    f"Before deduplication: "
    f"{before_dedup:,}"
)

print(
    f"After deduplication:  "
    f"{after_dedup:,}"
)

print(
    f"Duplicate records removed: "
    f"{before_dedup - after_dedup:,}"
)


print("\nREFERENCE TABLE SHAPE")
print("---------------------")
print(GT_PROCESS_STARTS.shape)


print("\nUNIQUE PROCESS TYPES")
print("--------------------")

print(
    GT_PROCESS_STARTS[
        "process_code"
    ].value_counts()
    .sort_index()
    .to_string()
)


print("\nFIRST 10 CLEAN PROCESS STARTS")
print("-----------------------------")

print(
    GT_PROCESS_STARTS.head(10)
    .to_string(index=False)
)


print("\nCASE ID CHECK")
print("-------------")

print(
    "Unique case IDs:",
    GT_PROCESS_STARTS["case_id"].nunique()
)

print(
    "Missing case IDs:",
    GT_PROCESS_STARTS["case_id"].isna().sum()
)

print("\nCLEAN GT PROCESS-START TABLE READY")

DATASET A — CLEAN PROCESS-START REFERENCE

PROCESS-START CLEANING
----------------------
Before deduplication: 1,819
After deduplication:  1,793
Duplicate records removed: 26

REFERENCE TABLE SHAPE
---------------------
(1793, 9)

UNIQUE PROCESS TYPES
--------------------
process_code
A    126
B     83
C    161
D    114
E    111
F    112
G    140
H    143
I    114
J     99
K    102
L     96
M    155
N    126
O    111

FIRST 10 CLEAN PROCESS STARTS
-----------------------------
                  run_id                           ts_utc                        timestamp           event current_process process_code process_name        case_id process_variant
theme_m1_20260630_175009 2026-06-30T12:21:12.794418+00:00 2026-06-30 12:21:12.794418+00:00 process_started               H            H       銀行勘定照合  BR-175009-001             NaN
theme_m1_20260630_175009 2026-06-30T12:21:41.335282+00:00 2026-06-30 12:21:41.335282+00:00 process_started               C            C    育児・産休申請確認  LA-17500

In [15]:
# Cell 12 — Inspect process lifecycle patterns

print("=" * 70)
print("DATASET A — PROCESS LIFECYCLE PATTERN ANALYSIS")
print("=" * 70)


# ------------------------------------------------------------
# Work with clean process starts
# ------------------------------------------------------------

starts = GT_PROCESS_STARTS.copy()

starts["timestamp"] = pd.to_datetime(
    starts["ts_utc"],
    utc=True,
    errors="coerce"
)


# ------------------------------------------------------------
# Get all lifecycle events
# ------------------------------------------------------------

lifecycle = GT_RAW[
    GT_RAW["event"].isin([
        "process_started",
        "process_switched_out",
        "process_suspended",
        "process_resumed"
    ])
].copy()

lifecycle["timestamp"] = pd.to_datetime(
    lifecycle["ts_utc"],
    utc=True,
    errors="coerce"
)

lifecycle = lifecycle.sort_values(
    ["run_id", "timestamp"],
    kind="stable"
).reset_index(drop=True)


# ------------------------------------------------------------
# Inspect event transitions
# ------------------------------------------------------------

print("\nGLOBAL LIFECYCLE EVENT TRANSITIONS")
print("----------------------------------")

transition_counts = Counter()

for run_id, group in lifecycle.groupby("run_id"):

    events = group["event"].tolist()

    for current, nxt in zip(events, events[1:]):
        transition_counts[(current, nxt)] += 1


for (current, nxt), count in transition_counts.most_common():
    print(
        f"{current:<25} -> "
        f"{nxt:<25} : {count:>5}"
    )


# ------------------------------------------------------------
# Inspect what usually follows process_started
# ------------------------------------------------------------

print("\n\nWHAT FOLLOWS process_started?")
print("=============================")

next_event_counts = Counter()

for run_id, group in lifecycle.groupby("run_id"):

    events = group["event"].tolist()

    for i, event in enumerate(events[:-1]):

        if event == "process_started":

            next_event_counts[
                events[i + 1]
            ] += 1


for event, count in next_event_counts.most_common():
    print(
        f"{event:<30} {count:>6}"
    )


# ------------------------------------------------------------
# Inspect what usually follows process_switched_out
# ------------------------------------------------------------

print("\n\nWHAT FOLLOWS process_switched_out?")
print("=================================")

next_after_switch = Counter()

for run_id, group in lifecycle.groupby("run_id"):

    events = group["event"].tolist()

    for i, event in enumerate(events[:-1]):

        if event == "process_switched_out":

            next_after_switch[
                events[i + 1]
            ] += 1


for event, count in next_after_switch.most_common():
    print(
        f"{event:<30} {count:>6}"
    )


print("\n\nLIFECYCLE PATTERN ANALYSIS COMPLETE")

DATASET A — PROCESS LIFECYCLE PATTERN ANALYSIS

GLOBAL LIFECYCLE EVENT TRANSITIONS
----------------------------------
process_started           -> process_switched_out      :  1415
process_switched_out      -> process_started           :  1413
process_started           -> process_started           :   248
process_switched_out      -> process_resumed           :   159
process_resumed           -> process_switched_out      :   157
process_started           -> process_suspended         :    98
process_suspended         -> process_started           :    96
process_resumed           -> process_resumed           :    29
process_switched_out      -> process_switched_out      :    18
process_suspended         -> process_resumed           :     2
process_suspended         -> process_suspended         :     1


WHAT FOLLOWS process_started?
process_switched_out             1415
process_started                   248
process_suspended                  98


WHAT FOLLOWS process_switched_out?
proces

In [16]:
# Cell 13 — Inspect process start → switch-out pairing

print("=" * 70)
print("DATASET A — PROCESS START / SWITCH-OUT PAIRING")
print("=" * 70)


# ------------------------------------------------------------
# Prepare lifecycle data
# ------------------------------------------------------------

lifecycle = GT_RAW[
    GT_RAW["event"].isin([
        "process_started",
        "process_switched_out",
        "process_suspended",
        "process_resumed"
    ])
].copy()

lifecycle["timestamp"] = pd.to_datetime(
    lifecycle["ts_utc"],
    utc=True,
    errors="coerce"
)

lifecycle = lifecycle.sort_values(
    ["run_id", "timestamp"],
    kind="stable"
).reset_index(drop=True)


# ------------------------------------------------------------
# Inspect switch-out records
# ------------------------------------------------------------

switches = lifecycle[
    lifecycle["event"] == "process_switched_out"
].copy()

print("\nTotal switch-out records:", len(switches))

print("\nSWITCH-OUT SCHEMA")
print("-----------------")

switch_columns = [
    "ts_utc",
    "current_process",
    "process_code",
    "process_name",
    "case_id",
    "process_variant"
]

switch_columns = [
    col for col in switch_columns
    if col in switches.columns
]

print(
    switches[switch_columns]
    .head(20)
    .to_string(index=False)
)


# ------------------------------------------------------------
# Inspect current_process values on switch-outs
# ------------------------------------------------------------

print("\n\nCURRENT PROCESS VALUES ON SWITCH-OUT")
print("-------------------------------------")

print(
    switches["current_process"]
    .value_counts(dropna=False)
    .to_string()
)


# ------------------------------------------------------------
# Compare each switch-out with the immediately
# preceding process_started
# ------------------------------------------------------------

print("\n\nSTART → SWITCH-OUT PAIRING CHECK")
print("================================")

pair_results = []

for run_id, group in lifecycle.groupby("run_id"):

    group = group.sort_values(
        "timestamp",
        kind="stable"
    )

    previous_start = None

    for _, row in group.iterrows():

        if row["event"] == "process_started":

            previous_start = row

        elif (
            row["event"] == "process_switched_out"
            and previous_start is not None
        ):

            gap_seconds = (
                row["timestamp"] -
                previous_start["timestamp"]
            ).total_seconds()

            pair_results.append({
                "run_id": run_id,
                "start_time": previous_start["timestamp"],
                "start_process": previous_start.get("process_code"),
                "start_case": previous_start.get("case_id"),
                "switch_time": row["timestamp"],
                "switch_process": row.get("current_process"),
                "gap_seconds": gap_seconds
            })

            previous_start = None


pairs = pd.DataFrame(pair_results)

print(
    f"\nCandidate start → switch pairs: "
    f"{len(pairs):,}"
)


# ------------------------------------------------------------
# Check whether process identities match
# ------------------------------------------------------------

if len(pairs) > 0:

    pairs["process_match"] = (
        pairs["start_process"].astype(str)
        ==
        pairs["switch_process"].astype(str)
    )

    print("\nPROCESS IDENTITY MATCH")
    print("----------------------")

    print(
        pairs["process_match"]
        .value_counts(dropna=False)
        .to_string()
    )


    print("\nPAIRING GAP STATISTICS (seconds)")
    print("--------------------------------")

    print(
        pairs["gap_seconds"]
        .describe()
        .to_string()
    )


    print("\nFIRST 20 CANDIDATE PAIRS")
    print("------------------------")

    print(
        pairs.head(20)
        .to_string(index=False)
    )


print("\n\nPAIRING ANALYSIS COMPLETE")

DATASET A — PROCESS START / SWITCH-OUT PAIRING

Total switch-out records: 1590

SWITCH-OUT SCHEMA
-----------------
                          ts_utc current_process process_code process_name case_id process_variant
2026-06-30T12:21:41.332221+00:00               H          NaN          NaN     NaN             NaN
2026-06-30T12:22:25.680580+00:00               C          NaN          NaN     NaN             NaN
2026-06-30T12:23:22.089303+00:00               B          NaN          NaN     NaN             reg
2026-06-30T12:23:46.001092+00:00               A          NaN          NaN     NaN             std
2026-06-30T12:24:20.921107+00:00               M          NaN          NaN     NaN             NaN
2026-06-30T12:24:53.806914+00:00               C          NaN          NaN     NaN             NaN
2026-06-30T12:26:32.486677+00:00               H          NaN          NaN     NaN             NaN
2026-06-30T12:26:55.497696+00:00               A          NaN          NaN     NaN          

In [17]:
# Cell 14 — Build GT active execution intervals

print("=" * 70)
print("DATASET A — BUILDING GT EXECUTION INTERVALS")
print("=" * 70)


# ------------------------------------------------------------
# Prepare lifecycle records
# ------------------------------------------------------------

lifecycle = GT_RAW[
    GT_RAW["event"].isin([
        "process_started",
        "process_switched_out",
        "process_suspended",
        "process_resumed"
    ])
].copy()

lifecycle["timestamp"] = pd.to_datetime(
    lifecycle["ts_utc"],
    utc=True,
    errors="coerce"
)

lifecycle = lifecycle.sort_values(
    ["run_id", "timestamp"],
    kind="stable"
).reset_index(drop=True)


# ------------------------------------------------------------
# Build intervals
# ------------------------------------------------------------

intervals = []

for run_id, group in lifecycle.groupby("run_id"):

    group = group.sort_values(
        "timestamp",
        kind="stable"
    ).reset_index(drop=True)

    active = None

    for _, row in group.iterrows():

        event = row["event"]

        # ----------------------------------------------
        # New process execution
        # ----------------------------------------------
        if event == "process_started":

            # If another process is active, do not overwrite it.
            # This can happen because processes may be interleaved.
            if active is None:

                active = {
                    "run_id": run_id,
                    "start_time": row["timestamp"],
                    "process_code": row.get("process_code"),
                    "process_name": row.get("process_name"),
                    "case_id": row.get("case_id"),
                    "process_variant": row.get("process_variant")
                }

        # ----------------------------------------------
        # Process switched out
        # ----------------------------------------------
        elif event == "process_switched_out":

            if active is not None:

                # Only close if the switched-out process
                # matches the active process.
                if str(active["process_code"]) == str(
                    row.get("current_process")
                ):

                    intervals.append({
                        **active,
                        "end_time": row["timestamp"],
                        "end_event": "process_switched_out"
                    })

                    active = None

        # ----------------------------------------------
        # Process suspended
        # ----------------------------------------------
        elif event == "process_suspended":

            if active is not None:

                if str(active["process_code"]) == str(
                    row.get("current_process")
                ):

                    intervals.append({
                        **active,
                        "end_time": row["timestamp"],
                        "end_event": "process_suspended"
                    })

                    active = None


# ------------------------------------------------------------
# Create DataFrame
# ------------------------------------------------------------

GT_INTERVALS_A = pd.DataFrame(intervals)


# ------------------------------------------------------------
# Calculate duration
# ------------------------------------------------------------

if not GT_INTERVALS_A.empty:

    GT_INTERVALS_A["duration_seconds"] = (
        GT_INTERVALS_A["end_time"]
        -
        GT_INTERVALS_A["start_time"]
    ).dt.total_seconds()


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nGT INTERVAL SUMMARY")
print("-------------------")

print(
    f"Intervals created: "
    f"{len(GT_INTERVALS_A):,}"
)

if not GT_INTERVALS_A.empty:

    print(
        f"Unique process executions represented: "
        f"{GT_INTERVALS_A['case_id'].nunique():,}"
    )

    print("\nEND EVENT COUNTS")
    print("----------------")

    print(
        GT_INTERVALS_A["end_event"]
        .value_counts()
        .to_string()
    )

    print("\nDURATION STATISTICS (seconds)")
    print("-----------------------------")

    print(
        GT_INTERVALS_A["duration_seconds"]
        .describe()
        .to_string()
    )

    print("\nFIRST 10 GT INTERVALS")
    print("---------------------")

    print(
        GT_INTERVALS_A[
            [
                "run_id",
                "start_time",
                "end_time",
                "process_code",
                "process_name",
                "case_id",
                "end_event",
                "duration_seconds"
            ]
        ]
        .head(10)
        .to_string(index=False)
    )

print("\nGT INTERVAL CONSTRUCTION COMPLETE")

DATASET A — BUILDING GT EXECUTION INTERVALS

GT INTERVAL SUMMARY
-------------------
Intervals created: 1,507
Unique process executions represented: 1,507

END EVENT COUNTS
----------------
end_event
process_switched_out    1409
process_suspended         98

DURATION STATISTICS (seconds)
-----------------------------
count    1507.000000
mean       45.874693
std        45.755835
min        17.500393
25%        26.414971
50%        34.852812
75%        51.083557
max      1144.824536

FIRST 10 GT INTERVALS
---------------------
                  run_id                       start_time                         end_time process_code process_name        case_id            end_event  duration_seconds
theme_m1_20260630_175009 2026-06-30 12:21:12.794418+00:00 2026-06-30 12:21:41.332221+00:00            H       銀行勘定照合  BR-175009-001 process_switched_out         28.537803
theme_m1_20260630_175009 2026-06-30 12:21:41.335282+00:00 2026-06-30 12:22:25.680580+00:00            C    育児・産休申請確認  LA-17500

In [18]:
# Cell 15 — Diagnose process starts not represented in GT intervals

print("=" * 70)
print("DATASET A — UNMATCHED PROCESS START DIAGNOSIS")
print("=" * 70)


# ------------------------------------------------------------
# All clean process starts
# ------------------------------------------------------------

all_starts = GT_PROCESS_STARTS.copy()

all_starts["timestamp"] = pd.to_datetime(
    all_starts["ts_utc"],
    utc=True,
    errors="coerce"
)


# ------------------------------------------------------------
# Starts already represented in our intervals
# ------------------------------------------------------------

matched_start_keys = set(
    zip(
        GT_INTERVALS_A["run_id"],
        GT_INTERVALS_A["start_time"],
        GT_INTERVALS_A["case_id"]
    )
)


all_starts["matched"] = [
    (
        row["run_id"],
        row["timestamp"],
        row["case_id"]
    ) in matched_start_keys
    for _, row in all_starts.iterrows()
]


unmatched = all_starts[
    ~all_starts["matched"]
].copy()


print("\nPROCESS START COVERAGE")
print("----------------------")

print("Total clean starts:", len(all_starts))
print("Matched to interval:", all_starts["matched"].sum())
print("Unmatched starts:", (~all_starts["matched"]).sum())


# ------------------------------------------------------------
# For each unmatched start, find the next lifecycle event
# ------------------------------------------------------------

lifecycle = GT_RAW[
    GT_RAW["event"].isin([
        "process_started",
        "process_switched_out",
        "process_suspended",
        "process_resumed"
    ])
].copy()

lifecycle["timestamp"] = pd.to_datetime(
    lifecycle["ts_utc"],
    utc=True,
    errors="coerce"
)

lifecycle = lifecycle.sort_values(
    ["run_id", "timestamp"],
    kind="stable"
).reset_index(drop=True)


diagnostics = []


for _, start in unmatched.iterrows():

    group = lifecycle[
        lifecycle["run_id"] == start["run_id"]
    ]

    future = group[
        group["timestamp"] > start["timestamp"]
    ]

    if len(future) > 0:

        next_row = future.iloc[0]

        diagnostics.append({
            "run_id": start["run_id"],
            "case_id": start["case_id"],
            "process_code": start["process_code"],
            "start_time": start["timestamp"],
            "next_event": next_row["event"],
            "next_time": next_row["timestamp"],
            "next_current_process": next_row.get(
                "current_process"
            ),
            "gap_seconds": (
                next_row["timestamp"] -
                start["timestamp"]
            ).total_seconds()
        })

    else:

        diagnostics.append({
            "run_id": start["run_id"],
            "case_id": start["case_id"],
            "process_code": start["process_code"],
            "start_time": start["timestamp"],
            "next_event": "NO_FUTURE_LIFECYCLE_EVENT",
            "next_time": pd.NaT,
            "next_current_process": None,
            "gap_seconds": None
        })


unmatched_diag = pd.DataFrame(diagnostics)


# ------------------------------------------------------------
# What follows the unmatched starts?
# ------------------------------------------------------------

print("\n\nNEXT EVENT AFTER UNMATCHED START")
print("================================")

print(
    unmatched_diag["next_event"]
    .value_counts(dropna=False)
    .to_string()
)


# ------------------------------------------------------------
# Check whether next event has same process identity
# ------------------------------------------------------------

print("\n\nNEXT EVENT + PROCESS IDENTITY")
print("=============================")

same_process = (
    unmatched_diag["process_code"].astype(str)
    ==
    unmatched_diag["next_current_process"].astype(str)
)

print(
    same_process.value_counts(dropna=False)
    .to_string()
)


# ------------------------------------------------------------
# Show examples
# ------------------------------------------------------------

print("\n\nFIRST 30 UNMATCHED STARTS")
print("=========================")

print(
    unmatched_diag
    .head(30)
    .to_string(index=False)
)


print("\n\nUNMATCHED START DIAGNOSIS COMPLETE")

DATASET A — UNMATCHED PROCESS START DIAGNOSIS

PROCESS START COVERAGE
----------------------
Total clean starts: 1793
Matched to interval: 1507
Unmatched starts: 286


NEXT EVENT AFTER UNMATCHED START
next_event
process_switched_out         165
NO_FUTURE_LIFECYCLE_EVENT     58
process_started               48
process_suspended             15


NEXT EVENT + PROCESS IDENTITY
True     228
False     58


FIRST 30 UNMATCHED STARTS
                  run_id        case_id process_code                       start_time                next_event                        next_time next_current_process  gap_seconds
theme_m1_20260630_175009  BR-175009-004            H 2026-06-30 12:28:02.650411+00:00      process_switched_out 2026-06-30 12:28:45.840345+00:00                    H    43.189934
theme_m1_20260630_175009 EXP-175009-002            G 2026-06-30 12:29:26.327121+00:00      process_switched_out 2026-06-30 12:29:48.734321+00:00                    G    22.407200
theme_m1_20260630_175009 SUP-1750